In [1]:
#03_data_preprocessing_feature_engineering

In [2]:
!git clone https://github.com/aymanberri/RealEstate_Analytics.git


Cloning into 'RealEstate_Analytics'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 167 (delta 91), reused 87 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 236.45 KiB | 4.30 MiB/s, done.
Resolving deltas: 100% (91/91), done.


In [2]:
%cd RealEstate_Analytics


/content/RealEstate_Analytics


In [3]:
!ls


app  architecture.md  data  notebooks  README.md  requirements.txt


In [4]:
!git status


On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


## Imports

In [5]:
import pandas as pd
import sqlite3
import numpy as np
import os


## Load data

We load the cleaned data that we saved earlier in the SQLite DB

In [16]:
DB_PATH = os.path.join("data", "database.db")

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM jvc_apartments", conn)
conn.close()

print("Loaded shape:", df.shape)

df.head()


Loaded shape: (951, 14)


,title,price,property_type,frequency,bedrooms,bathrooms,area,location,url,price_clean,price_yearly_aed,bedrooms_clean,bathrooms_clean,area_clean
0,1BR Apartment for Rent | Balcony & Pool | JVC,"74,000",apartment,yearly,1,2,905 sqft,"AAA Residence, JVC District 13, Jumeirah Villa...",https://www.bayut.com/property/details-9398172...,74000,74000,1.0,2.0,905.0
1,1 B/R with Balcony | Pool & Gym | JVC,"69,000",apartment,yearly,1,1,883 sqft,"Emerald Tower, JVC District 18, Jumeirah Villa...",https://www.bayut.com/property/details-4864285...,69000,69000,1.0,1.0,883.0
2,"Binghatti Phoenix, Jumeirah Village Circle, Dubai","89,990",apartment,yearly,1,2,826 sqft,"Binghatti Phoenix, JVC District 13, Jumeirah V...",https://www.bayut.com/property/details-1347054...,89990,89990,1.0,2.0,826.0
3,Converted into 2BR | Private Garden | Furnished,"140,000",apartment,yearly,1,2,"1,133 sqft","Signature Livings South, Signature Livings, JV...",https://www.bayut.com/property/details-1330702...,140000,140000,1.0,2.0,1133.0
4,Spacious 1Br | Prime Location | JVC,"75,000",apartment,yearly,1,2,925 sqft,"Reef Residence, JVC District 13, Jumeirah Vill...",https://www.bayut.com/property/details-1366409...,75000,75000,1.0,2.0,925.0


## Target


Before going into the feature engineering, I define the target first. My Target for this current project is the yearly price/rent of the property `price_yearly_aed`

Since we are predicting a number, this is a Regression problem.
_(There are two types of ML tasks, regression and classification, regression is when we predict a number, whereas classification is when we predict a category)_    

!!! Nothing derived from the target field should be fed into the model, like in our case the `price_per_sqft`, this is done to prevent data leakage.

In [7]:
TARGET = "price_yearly_aed"


## Feature Engineering

We organize the features as such:
- Base Numerical
  - `bedrooms_clean`
  - `bathrooms_clean`
  - `area_clean`

- Engineered numerical (What we'll create/handle)
  - `price_per_sqft`  (LEAKAGE — don’t include)
  - `area_per_bedroom`
  - `bathrooms_per_bedroom`
  - `log_area`
  - `log_price`

- Categorical
  - `property_type`
  - `building`

We start by area by bedroom, this tells us wether the property is spacious or crampy.

In [8]:
df["area_per_bedroom"] = df["area_clean"] / df["bedrooms_clean"]


Next, bathroom per bedroom. This gives a sense of luxury, because a 2BR property with 1 bathroom is not like a 2BR property with 2 bathrooms.

In [9]:
df["bathrooms_per_bedroom"] = df["bathrooms_clean"] / df["bedrooms_clean"]


### Log transforms (for skew)

Why logs?
- Some models assume features are linearly related to the prediction target. But in our case, real estate prices are almost never linear, they are skewed.
- For example, in my dataset, you can see some outliers that create a huge gap between the listings. These rare listings with the high rental price tend to dominate the learning in a way that isn't helpful.

What does log do?

```
  5000  → something like 8.5  
  10000 → 9.2  
  15000 → 9.6  
  200000 → 12.2  
  300000 → 12.6    
```
- compresses huge values
- spread out smaller values
- Balances the data

In [13]:
df["log_area"] = np.log1p(df["area_clean"])
df["log_price"] = np.log1p(df["price_yearly_aed"]) # This is for modeling, not for EDA!!


### Derive building features

_We extract structured features like building, district, and engineered variables from raw fields so the model can learn meaningful patterns from the data, and later we encode and use them as numeric inputs for training machine-learning models rather than feeding raw text, which models can't interpret directly._

In [20]:
# 1) Extract building (everything before "JVC District <number>")
df["building"] = df["location"].str.extract(r"^(.*?),\s*JVC District \d+", expand=False)
df["building"] = df["building"].fillna(df["location"].str.rsplit(",", n=2).str[0].str.strip())
print("Sample buildings:", df["building"].unique()[:5])

# 2️) Extract district (the part that contains "District")
df["district"] = df["location"].str.extract(r"(JVC District \d+)", expand=False).fillna("Unknown")
print("Sample districts:", df["district"].unique()[:5])

# 3️3) Extract community (second-to-last part before city, usually Dubai)
df["community"] = df["location"].str.split(",").str[-2].str.strip().fillna("Unknown")
print("Sample communities:", df["community"].unique()[:5])


Sample buildings: ['AAA Residence' 'Emerald Tower' 'Binghatti Phoenix'
 'Signature Livings South, Signature Livings' 'Reef Residence']
Sample districts: ['JVC District 13' 'JVC District 18' 'JVC District 10' 'JVC District 11'
 'JVC District 14']
Sample communities: ['Jumeirah Village Circle (JVC)']


## Handle categorical features

We have a problem, many unique building values.     
Having a high-cardinality feature creates many problems:
- **Exploding dimensions**: one-hot encoding creates hundreds of dummy columns, creating a sparse dataset. (slower training and more memory usage)
- **Overfitting**: models like tree-based ones can "memorize" rare categories that only appear in the training set instead of learning the general rules.


So we keep the top N building selection (based on listings) and group the rest to an "Other" category.

In [21]:
TOP_N_BUILDINGS = 20

top_buildings = (
    df["building"]
    .value_counts()
    .head(TOP_N_BUILDINGS)
    .index
)

df["building_grouped"] = df["building"].where(
    df["building"].isin(top_buildings),
    "Other"
)


## Encoding